# 3. Entrenamiento de modelos

Este notebook toma la tabla Delta con features ya preparadas, entrena y compara 5 variantes de modelos o configuraciones, y registra cada ejecución en MLflow utilizando el experimento `control-2`.

In [ ]:
from pathlib import Path

import mlflow
import mlflow.sklearn
import pandas as pd
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("control-2-entrenamiento")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .getOrCreate()
)

features_df = spark.table("silver_control_2")
pdf = features_df.toPandas()

X = pdf[[col for col in pdf.columns if col != 'target']]
y = pdf['target']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

mlflow.set_experiment("control-2")
print("Experimento MLflow configurado: control-2")

In [ ]:
models = [
    {
        "name": "logistic_regression",
        "estimator": LogisticRegression(max_iter=500, random_state=42),
        "params": {"C": 1.0, "solver": "lbfgs"},
    },
    {
        "name": "random_forest_100",
        "estimator": RandomForestClassifier(n_estimators=100, random_state=42),
        "params": {"n_estimators": 100, "max_depth": None},
    },
    {
        "name": "random_forest_200",
        "estimator": RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42),
        "params": {"n_estimators": 200, "max_depth": 8},
    },
    {
        "name": "gradient_boosting",
        "estimator": GradientBoostingClassifier(random_state=42),
        "params": {"n_estimators": 100, "learning_rate": 0.05},
    },
    {
        "name": "knn",
        "estimator": KNeighborsClassifier(n_neighbors=5),
        "params": {"n_neighbors": 5, "weights": "distance"},
    },
]

for index, config in enumerate(models, start=1):
    run_name = f"modelo-run-{index}"
    print(f"\n=== Ejecutando {run_name} ({config['name']}) ===")
    with mlflow.start_run(run_name=run_name) as run:
        model = config["estimator"]
        model.fit(X_train, y_train)

        predictions = model.predict(X_test)
        probability_predictions = model.predict_proba(X_test)[:, 1]

        acc = accuracy_score(y_test, predictions)
        f1 = f1_score(y_test, predictions, zero_division=0)
        roc_auc = roc_auc_score(y_test, probability_predictions)

        mlflow.log_params(config["params"])
        mlflow.log_metric("accuracy", acc)
        mlflow.log_metric("f1_score", f1)
        mlflow.log_metric("roc_auc", roc_auc)

        mlflow.sklearn.log_model(model, artifact_path="model")

        print(f"Run: {run_name}")
        print(f"Accuracy: {acc:.4f}")
        print(f"F1 Score: {f1:.4f}")
        print(f"ROC AUC: {roc_auc:.4f}")

print("\nEntrenamiento finalizado. Revisa la UI de MLflow para comparar los runs.")